# Eksperimen Model: Multi-Layer Perceptron (Deep Learning Dasar)
Jaringan Saraf Tiruan (ANN) yang memproses fitur MFCC 1D. Menggunakan TensorFlow/Keras untuk melacak pergerakan Akurasi dan Loss (Learning Curve).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow Version: {tf.__version__}')

### 1. Data Prep (StandardScaler sangat disarankan untuk Neural Network)

In [ ]:
X = np.load('X_features.npy')
y = np.load('y_labels.npy')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

import joblib
joblib.dump(scaler, 'models/scaler_mlp.pkl')

### 2. Membangun Arsitektur Keras

In [ ]:
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(8, activation='softmax') # 8 Kelas bahasa
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

### 3. Pelatihan Model dengan Early Stopping

In [ ]:
# Early stopping untuk mencegah Overfitting
es = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[es],
    verbose=1
)

### 4. Analisis Learning Curve

In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Kurva Loss (Makin kecil makin baik)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Kurva Akurasi (Makin besar makin baik)')
plt.legend()
plt.show()

### 5. Evaluasi & Confusion Matrix

In [ ]:
y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)

classes = ['horas', 'sampurasun', 'adilkatalino', 'wawawa', 'kulanuwun', 'tabea', 'apahabarpian', 'noise']
print(classification_report(y_test, y_pred, target_names=classes))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix - MLP Neural Network')
plt.show()

### 6. Simpan Model Keras (.h5)

In [ ]:
model.save('models/mlp_model.h5')
print('Model MLP berhasil disimpan!')